# Experimento: FT-Transformer, fase 5 (triagem e Optuna)

**De onde partimos.** Na fase 4, as três features do MLP fizeram o FT ganhar do trial 0 pela primeira vez: `f_trial0_tres` teve **0,5506** no placar, contra 0,5530 do MLP adotado. Depois de quatro fases, a lição é que o FT reage aos **dados e à forma como eles viram tokens**, e não à regularização.

## 1. Triagem: o que nunca foi testado

Uma mudança por vez sobre `f_trial0_tres`, no placar comum, com veredito contra ela (`vs_base`) e contra o trial 0:

| Eixo | Células | Por quê |
|---|---|---|
| Oversampling | sem | Sem oversampling, a fase 4 quase empatou com a melhor versão (0,5503) |
| Taxa de aprendizado | 3e-4, 1e-3 | Ficou em 1e-4 (o padrão do artigo) em todas as fases |
| Agenda do lr | cosseno com aquecimento de 5 épocas (lr 1e-4 e 3e-4) | Com lr fixo, a rede para por volta da época 7 |
| Cabeças de atenção | 2, 4 | Não mudam o número de parâmetros, só como o `d_token` é repartido. Servem de controle |
| FFN | fator 2 | A largura da camada interna |
| **Tokens categóricos** | um token por variável | Hoje `Contract` vira 3 tokens e `PaymentMethod` vira 4. No artigo, cada variável categórica é um token com uma tabela de embeddings: 31 → 24 tokens |
| **Embeddings periódicos (PLR)** | σ = 0,1 e 1 | `tenure` entra como uma reta (x·W + b). As features que ajudaram são cortes em `tenure`, e o PLR deixa a rede aprender esses cortes (Gorishniy et al., 2022) |
| Features extras | as 9 candidatas do MLP, uma de cada vez, somadas às três | O FT reagiu às features mais do que o MLP |
| Média de 3 redes por dobra | sementes diferentes, média das probabilidades | Reduz o ruído da parada |

## 2. Optuna

O Optuna procura nos eixos da triagem, junto com a arquitetura (blocos, `d_token`), a regularização (três dropouts, `wd`) e o lote. Das features extras, entram como sim/não só as que tiveram sinal positivo na triagem.

**Sem escolher nas dobras do placar.** Cada trial vale o KS médio em **15 dobras próprias** (sementes 201 a 203), e não nas sementes 101 a 103 do placar. O trial 0 é a própria base. No fim, os 3 melhores trials (e o melhor com a média de 3 redes) são medidos no placar. O estudo fica em sqlite na pasta da execução: retomar a execução retoma o estudo.

## 3. Média com o MLP

As predições OOF do FT e do MLP vêm das mesmas dobras do placar. A média das duas é medida contra cada uma.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_fase5.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` (**troque: o da fase 4 estava inválido**) e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*. São cerca de 7 h de GPU, e com `HORAS_LIMITE` a execução para limpa. Para continuar, use `RUN_NAME` com o mesmo nome e `RESUME_FROM = "github"`.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-fase5-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE4_RUN = "ft-fase4-20261117-0037"   # execução da fase 4 na branch de resultados (ou uma pasta local): traz as fases 1 a 4 já medidas
N_TRIALS = 60                   # trials do Optuna (cada um mede 15 dobras próprias)
TOP_OPTUNA = 3                  # melhores trials que vão para o placar
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-fase5-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    N_TRIALS, TOP_OPTUNA = 3, 1
    print("MODO_TESTE: 3 épocas, seis células da triagem e 3 trials. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2, fase4 as F4, fase5 as F5

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase=5, dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas das fases 1 a 4 (com o trial 0 e a base f_trial0_tres) entram aqui sem treinar de novo
import tempfile
if FASE4_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE4_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE4_RUN) else \
            fluxo.baixar_github(_tmp, FASE4_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fases 1 a 4: {len(anteriores)} linhas trazidas de {FASE4_RUN}" if anteriores else "Fases 1 a 4: nada trazido; as bases serão medidas aqui")


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F1.tabela(linhas, campos).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")
Xf_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5], features=F4.TRES)
print(f"com as features do MLP: {Xf_.shape[1]} tokens + [CLS] ({', '.join(F4.TRES)})")
for nome, hp in (("tokens categóricos", dict(tokens="cat")), ("PLR", dict(num_emb="plr"))):
    g_, n_ = M.estrutura(hp, list(Xf_.columns), np.asarray(Xf_, dtype="float32"))
    print(f"{nome}: {len(g_)} tokens | colunas com PLR: {[Xf_.columns[j] for j in n_]}")

# 1. Triagem: uma mudança por vez sobre `f_trial0_tres`

Vale olhar:
- **`vs_base`:** o veredito contra `f_trial0_tres`. É ele que diz se a mudança ajudou;
- **`epocas` e `minutos`:** o lr maior e a agenda mudam onde a parada cai;
- **as features extras:** as que tiverem delta positivo entram como eixo do Optuna.

A figura mostra o delta contra a base com o intervalo de 95% de cada célula. Uma barra que não cruza o zero é um veredito.

In [ ]:
T0, B = ("ft_transformer", "trial0"), ("ft_transformer", F5.ROT_BASE)
TRIAGEM = F5.grade_triagem()
if MODO_TESTE:
    TRIAGEM = {r: g for r, g in TRIAGEM.items() if r in ("t5_sem_over", "t5_tok_cat", "t5_plr_s1", "t5_cos_lr3e-4", "t5_mais_log_tenure", "t5_3redes")}
assert MODO_TESTE or ("trial0" in anteriores and F5.ROT_BASE in anteriores), "faltam o trial 0 ou a base no placar trazido"
itens = [("trial0", F1.TRIAL0, None), (F5.ROT_BASE, F5.BASE, {"vs_trial0": T0})]
itens += [(r, c, {"vs_trial0": T0, "vs_base": B}) for r, (_, c) in TRIAGEM.items()]
t0 = time.time()
linhas = medir(itens)
print(f"{(time.time() - t0) / 60:.1f} min")
tri = {r["rotulo"]: r for r in linhas if r["rotulo"] in TRIAGEM}
tt = F1.tabela(list(tri.values()), ("vs_trial0", "vs_base")).drop(columns=["blocos", "d_token"])
tt.insert(1, "eixo", [TRIAGEM[r][0] for r in tt.rotulo])
tt["delta_base"] = [tri[r]["vs_base"]["delta"] for r in tt.rotulo]
display(tt.sort_values("delta_base", ascending=False).round(4))

# delta contra a base, com o IC95 do bootstrap pareado
ordem = sorted(tri, key=lambda r: tri[r]["vs_base"]["delta"])
fig, ax = plt.subplots(figsize=(8, .32 * len(ordem) + 1.2), facecolor=F2.FUNDO)
ax.set_facecolor(F2.FUNDO)
for i, r in enumerate(ordem):
    v = tri[r]["vs_base"]
    cor = F2.AZUL if v["veredito"] == "ganhou" else F2.LARANJA if v["veredito"] == "perdeu" else F2.SUAVE
    ax.plot(v["ic95"], [i, i], color=cor, lw=2, solid_capstyle="round")
    ax.plot([v["delta"]], [i], "o", color=cor, ms=6, mec=F2.FUNDO, mew=1.5)
ax.axvline(0, color=F2.GRADE, lw=1)
ax.set_yticks(range(len(ordem)), ordem, fontsize=8, color=F2.TINTA)
ax.set_xlabel(f"KS OOF: diferença contra {F5.ROT_BASE} (IC 95%)", color=F2.SUAVE, fontsize=9)
ax.tick_params(colors=F2.SUAVE, labelsize=8)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.set_title("Triagem: uma mudança por vez (azul = ganhou, laranja = perdeu)", color=F2.TINTA, fontsize=10, loc="left")
fig.tight_layout(); fig.savefig(os.path.join(PASTA, "triagem.png"), dpi=150, facecolor=F2.FUNDO); plt.show()
_ = salvar()

# 2. Optuna

As features extras com delta positivo na triagem viram eixos de sim/não. A lista fica salva na pasta, então retomar a execução não muda o espaço de busca no meio do estudo.

Cada trial é medido em 15 dobras **próprias** (sementes 201 a 203). O placar não é usado aqui.

In [ ]:
import optuna
arq_extras = os.path.join(PASTA, "optuna_extras.json")
if os.path.isfile(arq_extras):
    EXTRAS_OK = json.load(open(arq_extras))
else:
    EXTRAS_OK = [f for f in F5.EXTRAS if f"t5_mais_{f}" in tri and tri[f"t5_mais_{f}"]["vs_base"]["delta"] > 0]
    json.dump(EXTRAS_OK, open(arq_extras, "w"))
print("Features extras no Optuna:", EXTRAS_OK or "nenhuma")
D_OPT = F5.divs_optuna(F5.SEMENTES_OPTUNA[:1] if MODO_TESTE else F5.SEMENTES_OPTUNA)


def ao_trial(st, t, r):
    if wandb_run is not None:
        wandb_run.log({"optuna/trial": t.number, "optuna/ks": r["ks_kfold"], "optuna/melhor": st.best_value, "optuna/minutos": r.get("tempo_min")})
    salvar()


t0 = time.time()
estudo = F5.estudar(PASTA, N_TRIALS, EXTRAS_OK, medir=lambda c, D: F1.medir(c, DISPOSITIVOS, POR_GPU, D=D),
                    continuar=continuar, ao_trial=ao_trial, D=D_OPT)
print(f"{(time.time() - t0) / 60:.1f} min | {len(estudo.trials)} trials")
df_opt = estudo.trials_dataframe().sort_values("value", ascending=False)
df_opt.to_csv(os.path.join(PASTA, "optuna_trials.csv"), index=False)
display(df_opt.head(10).round(4))
res_opt = F5.resumo_optuna(estudo)
res_opt.to_csv(os.path.join(PASTA, "optuna_resumo.csv"), index=False)
display(res_opt.round(4))
try:
    imp = optuna.importance.get_param_importances(estudo)
    print("Importância dos eixos (fANOVA):"); display(pd.Series(imp).round(3).to_frame("importância"))
    json.dump(imp, open(os.path.join(PASTA, "optuna_importancia.json"), "w"), indent=1)
except Exception as e:
    print("importância indisponível:", e)
_ = salvar()

# 3. Os melhores trials no placar

Os `TOP_OPTUNA` melhores trials (sem repetir configuração, e sem a própria base) são medidos no placar, contra o trial 0 e contra a base. O melhor também é medido com a média de 3 redes por dobra.

In [ ]:
tops = [t for t in F5.melhores(estudo, TOP_OPTUNA + 1) if F1._chave(F5.config_do_trial(t.params, EXTRAS_OK)) != F1._chave(F5.BASE)][:TOP_OPTUNA]
itens_o = [(f"o5_top{i + 1}", F5.config_do_trial(t.params, EXTRAS_OK), {"vs_trial0": T0, "vs_base": B}) for i, t in enumerate(tops)]
if itens_o:
    c1 = itens_o[0][1]
    itens_o.append(("o5_top1_3redes", dict(hp=dict(c1["hp"], n_redes=3), dados=dict(c1["dados"])), {"vs_trial0": T0, "vs_base": B}))
for (r, c, _), t in zip(itens_o, tops + tops[:1]):
    print(r, f"| trial {t.number} | KS Optuna {t.value:.4f} |", json.dumps(c, sort_keys=True))
linhas_o = medir(itens_o)
display(F1.tabela(linhas_o, ("vs_trial0", "vs_base")).round(4))

# 4. Média com o MLP

O MLP adotado vem de `resultados/` (versionado no repositório). Do FT entra a versão de maior KS do placar. Como as predições OOF das duas redes vêm das mesmas dobras, a média pode ser medida no placar sem treinar nada. São duas formas:
- **média das probabilidades;**
- **média dos postos:** cada modelo vira um ranking de 0 a 1, o que não depende da calibração de nenhum deles.

In [ ]:
RES = os.path.join(REPO_DIR, "resultados")
rot_mlp = json.load(open(os.path.join(RES, "escolhidos.json")))["mlp"]["rotulo"]
mlp = {r["rotulo"]: r for r in comum.ler_placar(RES) if r["modelo"] == "mlp"}[rot_mlp]
fts = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
ft = max(fts, key=lambda r: r["ks_kfold"])
assert mlp["assinatura"] == ft["assinatura"] == F1.assinatura() and mlp["sementes"] == ft["sementes"]
y_tr = np.asarray(F1._dados()[2])
oof_mlp, oof_ft = np.load(os.path.join(RES, mlp["oof"])), np.load(os.path.join(PASTA, ft["oof"]))
D_pl, sem = F1.divs(), sorted({d["semente"] for d in F1.divs()})
medias, linhas_m = {}, []
for tipo, E in F5.media_com_mlp(y_tr, oof_ft, oof_mlp).items():
    med = comum.resumir_placar(y_tr, D_pl, [E[sem.index(d["semente"])][d["validacao"]] for d in D_pl])
    medias[tipo] = dict(ks_kfold=med["ks_kfold"], ep=med["ep"], ks_oof=med["ks_oof"], auc_oof=med["auc_oof"],
                        vs_mlp=comum.comparar_placar(y_tr, oof_mlp, E), vs_ft=comum.comparar_placar(y_tr, oof_ft, E))
    np.save(os.path.join(PASTA, f"media_mlp_ft_{tipo}.npy"), E)
for nome, r in ((f"MLP {rot_mlp}", mlp), (f"FT {ft['rotulo']}", ft)):
    linhas_m.append(dict(versao=nome, ks_kfold=r["ks_kfold"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"]))
vv = lambda v: f"{v['veredito']} {v['delta']:+.4f} [{v['ic95'][0]:+.4f}, {v['ic95'][1]:+.4f}]"
for tipo, m in medias.items():
    linhas_m.append(dict(versao=f"média ({tipo})", ks_kfold=m["ks_kfold"], ks_oof=m["ks_oof"], auc_oof=m["auc_oof"],
                         vs_mlp=vv(m["vs_mlp"]), vs_ft=vv(m["vs_ft"])))
display(pd.DataFrame(linhas_m).round(4))
json.dump(dict(mlp=rot_mlp, ft=ft["rotulo"], **medias), open(os.path.join(PASTA, "media_mlp_ft.json"), "w"), indent=1, default=float)
_ = salvar()

# 5. Placar das fases 1 a 5

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
tab = F1.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]), ("vs_trial0", "vs_base"))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_ft_transformer_fase5.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do trial 0: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 5

- **Triagem: o que ganhou ou perdeu da base?** _…_
- **Tokens categóricos e PLR mudaram alguma coisa?** _…_
- **Optuna: quais eixos importaram, e o melhor trial se confirmou no placar?** _…_
- **A média com o MLP ganhou das duas redes?** _…_
- **Versão do FT a adotar e o próximo passo:** _…_